# `share_dispersion` Recalibration — `early_access_brazil` (2050)

**Why a separate pass:** `early_access_brazil`'s pass-1 solve inherited `share_dispersion` from
`early_access_2050`'s own calibration. That value no longer bites: the Brazil import changed
`offre_locale_hors_TECH_HS` in C1/C4 (cheap import competes with `TECH_HS` too, see the previous
report), so home systems fell to 0% (C2) and 44% (C4) of target instead of holding near the
calibrated ~95-100% seen in `early_access`. Recalibrating on `early_access_brazil`'s own pass-1
output, same formula/denominator as every prior pass, targets unchanged:
C1 2.079, C2 0.132, C3 0.648, C4 1.543, C5 0.000 GWh/y. C2 and C5 forced to 0 after the formula,
same as always.


In [1]:
import pandas as pd
import json

ROOT = "../../../EnergyScope_BO_nord_amazonia"
CS_ROOT_2050 = f"{ROOT}/case_studies/C1_C2_C3_C4_C5"
DISPERSED_DEMAND_GWH_BY_CLUSTER = {1: 2.079, 2: 0.132, 3: 0.648, 4: 1.543, 5: 0.000}
EXCLUDE_TECHS_FOR_OFFRE_LOCALE = {"ELECTRICITY", "PV_HS", "HS_DIESEL", "BATT_HS"}

cs_outputs = f"{CS_ROOT_2050}/norte_amazonia_early_access_brazil_2050/outputs"

solve_info = pd.read_csv(f"{cs_outputs}/Solve_info.csv", sep=r"\t;\t", header=None,
                          index_col=0, engine="python")
solve_result_num = int(float(solve_info.loc["solve_result_num", 1]))
assert solve_result_num == 0, (
    f"early_access_brazil_2050: solve_result_num={solve_result_num} != 0 -- "
    f"refusing to calibrate share_dispersion from a non-optimal pass-1 solve")

cs_dir = f"{cs_outputs}/regional_results"
yb = pd.read_csv(f"{cs_dir}/Year_balance.csv", sep=";")
yb["ELECTRICITY"] = pd.to_numeric(yb["ELECTRICITY"], errors="coerce")
yb_tech = yb[~yb["Elements"].isin(EXCLUDE_TECHS_FOR_OFFRE_LOCALE)]
yb_tech = yb_tech[yb_tech["ELECTRICITY"] > 1e-9]
tech_sum = yb_tech.groupby("Regions")["ELECTRICITY"].sum()

res = pd.read_csv(f"{cs_dir}/Resources.csv", sep=";")
res_elec = res[res["Resources"] == "ELECTRICITY"].set_index("Regions")
r_local = pd.to_numeric(res_elec["R_year_local"], errors="coerce").fillna(0.0)
r_ext = pd.to_numeric(res_elec["R_year_exterior"], errors="coerce").fillna(0.0)
# R_year_import / R_year_export deliberately excluded -- inter-cluster exchange, not local supply.
# NOTE: for C3/C5 this R_year_exterior now includes the Brazil import itself (76.004 / 37.796 GWh)
# -- correctly folded into the denominator, since it IS local (non-TECH_HS) supply from this
# cluster's own Resources.csv perspective, same treatment as any other exterior resource.

shares, offres = {}, {}
for k in range(1, 6):
    region = f"C{k}"
    offre_locale = (float(tech_sum.get(region, 0.0))
                     + float(r_local.get(region, 0.0)) + float(r_ext.get(region, 0.0)))
    target = DISPERSED_DEMAND_GWH_BY_CLUSTER[k]
    share = target / (target + offre_locale) if (target + offre_locale) > 0 else 0.0
    shares[k] = share
    offres[k] = offre_locale

print(f"early_access_brazil_2050 (solve_result_num={solve_result_num}):")
print(f"  offre_locale_hors_TECH_HS = {offres}")
print(f"  share_dispersion (raw)    = {shares}")

DEPLOYED_SHARE_DISPERSION_BRAZIL = dict(shares)
DEPLOYED_SHARE_DISPERSION_BRAZIL[2] = 0.0  # forced to 0, same as every prior pass
DEPLOYED_SHARE_DISPERSION_BRAZIL[5] = 0.0  # target is 0, forced explicitly for clarity
print()
print(f"Deployed (C2/C5 forced to 0): {DEPLOYED_SHARE_DISPERSION_BRAZIL}")

print()
print("=== Comparison against the inherited early_access value (no longer used) ===")
for k in range(1, 6):
    old = json.load(open(f"{ROOT}/Data/2050/early_access/C{k}/Misc.json"))["share_dispersion"]
    print(f"C{k}: inherited(early_access)={old:.6f}  recalibrated(brazil)={DEPLOYED_SHARE_DISPERSION_BRAZIL[k]:.6f}")


early_access_brazil_2050 (solve_result_num=0):
  offre_locale_hors_TECH_HS = {1: 36.59147474458595, 2: 3.623624677080471, 3: 224.81673575977572, 4: 50.29543079228892, 5: 120.88163010539934}
  share_dispersion (raw)    = {1: 0.053761946646157216, 2: 0.03514728210344317, 3: 0.0028740636437727444, 4: 0.029765561503638814, 5: 0.0}

Deployed (C2/C5 forced to 0): {1: 0.053761946646157216, 2: 0.0, 3: 0.0028740636437727444, 4: 0.029765561503638814, 5: 0.0}

=== Comparison against the inherited early_access value (no longer used) ===
C1: inherited(early_access)=0.050638  recalibrated(brazil)=0.053762
C2: inherited(early_access)=0.000000  recalibrated(brazil)=0.000000
C3: inherited(early_access)=0.003182  recalibrated(brazil)=0.002874
C4: inherited(early_access)=0.012949  recalibrated(brazil)=0.029766
C5: inherited(early_access)=0.000000  recalibrated(brazil)=0.000000


## Deploy to `Misc.json`, with assertion (brazil only)

In [2]:
for k in range(1, 6):
    target = f"{ROOT}/Data/2050/early_access_brazil/C{k}/Misc.json"
    deployed_value = DEPLOYED_SHARE_DISPERSION_BRAZIL[k]

    with open(target, encoding="utf-8") as f:
        misc = json.load(f)
    misc["share_dispersion"] = deployed_value
    with open(target, "w", encoding="utf-8") as f:
        json.dump(misc, f, indent=4)

    with open(target, encoding="utf-8") as f:
        check = json.load(f)
    actual = check["share_dispersion"]
    if abs(actual - deployed_value) > 1e-9:
        raise AssertionError(f"{target}: share_dispersion = {actual}, expected {deployed_value}")
print("OK -- share_dispersion deployed and verified in Misc.json (C1-C5) for early_access_brazil_2050")


OK -- share_dispersion deployed and verified in Misc.json (C1-C5) for early_access_brazil_2050


## Reprint `reg_misc.dat` from the deployed catalog (no solve)

In [3]:
import sys
sys.path.insert(0, r"C:\Valen\Tfe\EnergyScope_BO_nord_amazonia")
from pathlib import Path
from esmc import Esmc

case_study = "norte_amazonia_early_access_brazil_2050"
FT_TO_DROP = ['BIOMASS_TO_GASOLINE', 'BIOMASS_TO_DIESEL', 'BIOWASTE_TO_GASOLINE', 'BIOWASTE_TO_DIESEL',
              'POWER_TO_GASOLINE', 'POWER_TO_DIESEL', 'H2_TO_GASOLINE', 'H2_TO_DIESEL']
AMPL_PATH = r'C:\Users\valen\AMPL'

config = {'case_study': case_study, 'comment': 'share_dispersion recalibration reprint (no solve)',
          'regions_names': ['C1', 'C2', 'C3', 'C4', 'C5'],
          'gwp_limit_overall': None, 're_share_primary': None, 'f_perc': True,
          'year': 2050, 'scenario': 'early_access_brazil'}

my_model = Esmc(config, nbr_td=16)
current_project = Path(r"C:\Valen\Tfe\EnergyScope_BO_nord_amazonia")
my_model.project_dir = current_project
my_model.dat_dir = current_project / 'case_studies' / my_model.space_id / '00_td_dat'
my_model.cs_dir = current_project / 'case_studies' / my_model.space_id / case_study
my_model.dat_dir.mkdir(parents=True, exist_ok=True)
my_model.cs_dir.mkdir(parents=True, exist_ok=True)

my_model.read_data_indep()
my_model.init_regions()

my_model.ref_region.data['Technologies'] = my_model.ref_region.data['Technologies'].drop(index=FT_TO_DROP)
my_model.data_indep['Layers_in_out'] = my_model.data_indep['Layers_in_out'].drop(index=FT_TO_DROP)
for r_code, region in my_model.regions.items():
    region.data['Technologies'] = region.data['Technologies'].drop(index=FT_TO_DROP)

assert case_study.startswith('norte_amazonia_early_access_')
for r_code, region in my_model.regions.items():
    region.data['Technologies'].loc['PV_UTILITY', 'f_max'] = 1e15
    region.data['Technologies'].loc['BATT_LI', 'f_max'] = 1e15

for k in range(1, 6):
    region_code = f"C{k}"
    in_memory_sd = float(my_model.regions[region_code].data['Misc']['share_dispersion'])
    expected = DEPLOYED_SHARE_DISPERSION_BRAZIL[k]
    assert abs(in_memory_sd - expected) < 1e-9, (
        f"{region_code}: in-memory share_dispersion={in_memory_sd} after init_regions(), "
        f"expected {expected} (deployed Misc.json)")

my_model.init_ta(algo='read', ampl_path=AMPL_PATH)
my_model.print_td_data()
my_model.print_data(indep=True)

print(f"OK -- reg_misc.dat reprinted for {case_study} at {my_model.cs_dir} "
      f"(pre-print in-memory check passed, no solve)")

dat_path = my_model.cs_dir / "reg_misc.dat"
lines = dat_path.read_text(encoding="utf-8").splitlines()
header_line_idx = next(i for i, l in enumerate(lines)
                        if l.strip().startswith("param") and "share_dispersion" in l)
header_fields = lines[header_line_idx].split()
col_idx = header_fields.index("share_dispersion") - 1
for k in range(1, 6):
    row = next((l for l in lines[header_line_idx+1:] if l.split() and l.split()[0] == f"C{k}"), None)
    assert row is not None, f"reg_misc.dat: no C{k} row found in the share_dispersion block"
    printed_sd = float(row.split()[col_idx])
    expected = DEPLOYED_SHARE_DISPERSION_BRAZIL[k]
    assert abs(printed_sd - expected) < 1e-6, (
        f"reg_misc.dat C{k}: printed share_dispersion={printed_sd}, expected {expected}")
print("OK -- reg_misc.dat text spot-check passed for all 5 clusters")


[INFO    ] (read_data_indep): Read indep data from C:\Valen\Tfe\EnergyScope_BO_nord_amazonia\Data\2050\early_access_brazil\00_INDEP


[INFO    ] (init_regions): Initialising regions: C1, C2, C3, C4, C5


[INFO    ] (read_data): Read data from C:\Valen\Tfe\EnergyScope_BO_nord_amazonia\Data\2050\early_access_brazil\02_REF_REGION


[INFO    ] (read_data): Read data from C:\Valen\Tfe\EnergyScope_BO_nord_amazonia\Data\2050\early_access_brazil\C1


[INFO    ] (read_data): Read data from C:\Valen\Tfe\EnergyScope_BO_nord_amazonia\Data\2050\early_access_brazil\C2


...

[INFO    ] (generate_t_h_td): t_h_td and td_count generated


[INFO    ] (print_td_data): Printing TD data into C:\Valen\Tfe\EnergyScope_BO_nord_amazonia\case_studies\C1_C2_C3_C4_C5\norte_amazonia_early_access_brazil_2050\reg_16TD.dat


[INFO    ] (generate_t_h_td): t_h_td and td_count generated


[INFO    ] (print_data): Printing regional data into C:\Valen\Tfe\EnergyScope_BO_nord_amazonia\case_studies\C1_C2_C3_C4_C5\norte_amazonia_early_access_brazil_2050


OK -- reg_misc.dat reprinted for norte_amazonia_early_access_brazil_2050 at C:\Valen\Tfe\EnergyScope_BO_nord_amazonia\case_studies\C1_C2_C3_C4_C5\norte_amazonia_early_access_brazil_2050 (pre-print in-memory check passed, no solve)
OK -- reg_misc.dat text spot-check passed for all 5 clusters
